# Validate against census flows

**Question.** Do the home-to-work flows that I measure from trajectories agree with an official table of flows, and which distance decay do they imply?

**You will get** home and work places from trajectories, expansion weights for a biased sample, the agreement of weighted and unweighted flows with a reference table (common part of commuters, R-squared, Poisson deviance), and a gravity model with a confidence interval for its distance exponent. The last section repeats the gravity and radiation comparison on real check-ins.

**Prerequisites.** [Measure individual mobility](measure-individual-mobility.md) and [Flows between places](flows-between-places.md).

**Data.** First a synthetic region of 36 zones with a *known* answer: a reference ("census") table of commuting flows generated by a gravity model with distance exponent -1.8, and a sample of 27,000 people from it, with a participation rate that grows with the zone number (a biased sample). Official tables such as INSEE's mobility files or the US commuting flows have the same shape (an origin, a destination and a count) and can replace the synthetic table. Then [Brightkite](https://snap.stanford.edu/data/loc-brightkite.html) check-ins, the first 300,000 rows downloaded with `fastmob.data.load_brightkite`.

In [1]:
%pip install -q 'fastmob>=0.3.4' pandas statsmodels

## A reference table with a known answer

We lay out 36 zones on a 6 by 6 grid (about 13 km apart), give each a number of resident workers and of jobs, and generate the reference flows with a singly constrained gravity model: the flow from an origin is split among destinations in proportion to their jobs and to `distance ** -1.8`. Every person in the sample below is drawn from this table, so we can check what the analysis recovers.

In [2]:
import fastmob
import numpy as np
import pandas as pd
from fastmob.core import FlowDataFrame, Locations
from fastmob.measures.evaluation import compare_flow_models
from fastmob.models import Gravity

rng = np.random.default_rng(0)
side = 6
zone = np.arange(side * side)
zones = pd.DataFrame(
    {
        "tile_id": zone,
        "lat": 45.0 + 0.12 * (zone // side),
        "lng": 7.0 + 0.16 * (zone % side),
        "relevance": rng.integers(2_000, 40_000, side * side),  # jobs
        "tot_outflow": rng.integers(20_000, 80_000, side * side),  # resident workers
    }
)
locations = Locations.from_tessellation(
    zones, tile_id_col="tile_id", lat_col="lat", lng_col="lng"
)

truth = Gravity(gravity_type="singly constrained", deterrence_func_args=[-1.8])
census = truth.generate(locations, out_format="flows_sample", random_state=1)
reference = census.to_pandas()
print(
    len(reference),
    "origin-destination pairs,",
    int(reference["flow"].sum()),
    "commuters",
)
reference.nlargest(3, "flow")

1260 origin-destination pairs, 1836919 commuters


,origin,destination,flow
606,17,11,15692.0
1152,32,33,15060.0
648,18,19,14748.0


The reference table has 1,260 pairs of zones (36 zones, every pair but the diagonal) and about 1.84 million commuters; the largest flows are around 15,000 commuters. The table is on the scale of a whole region, so a sample of a few percent has to be scaled up before it can be compared in absolute terms.

## A biased sample of people

A real data set covers a few percent of the population, and not evenly. We draw each commuter of the reference table into the sample with a probability that grows from 0.4% in the first zone to 2.4% in the last, then give each sampled person fixed home and work spots in their zones and six days of records: a night record at home (23:00) and a day record at work (10:00).

In [3]:
participation = 0.004 + 0.02 * zone / zone.max()
drawn = rng.binomial(reference["flow"].astype(int), participation[reference["origin"]])
people = reference.loc[reference.index.repeat(drawn), ["origin", "destination"]]
people = people.reset_index(drop=True).assign(uid=lambda d: d.index)


def spot(zone_of_person):
    return (
        45.0 + 0.12 * (zone_of_person // side) + rng.uniform(-0.03, 0.03, len(people)),
        7.0 + 0.16 * (zone_of_person % side) + rng.uniform(-0.04, 0.04, len(people)),
    )


home_lat, home_lng = spot(people["origin"].to_numpy())
work_lat, work_lng = spot(people["destination"].to_numpy())
monday = pd.Timestamp("2024-03-04")
records = []
for day in range(6):
    for hour, lat, lng in ((23, home_lat, home_lng), (10, work_lat, work_lng)):
        when = monday + pd.Timedelta(days=day, hours=hour)
        records.append(
            pd.DataFrame(
                {"uid": people["uid"], "datetime": when, "lat": lat, "lng": lng}
            )
        )
trajectories = pd.concat(records, ignore_index=True)
print(len(people), "people,", len(trajectories), "records")

27420 people, 329040 records


## Home and work from trajectories

`home_location` is the most visited place at night and `work_location` the most visited place during working hours on weekdays. We map each place to its zone (the grid makes this a rounding) and count the people for each pair of zones. People whose home and work fall in the same zone are not commuters between zones and are left out, as in the reference table.

In [4]:
home = fastmob.home_location(trajectories)
work = fastmob.work_location(trajectories)


def zone_of(places):
    row = np.rint((places["lat"] - 45.0) / 0.12).astype(int).clip(0, side - 1)
    col = np.rint((places["lng"] - 7.0) / 0.16).astype(int).clip(0, side - 1)
    return (row * side + col).to_numpy()


pairs = home.assign(home=zone_of(home))[["uid", "home"]].merge(
    work.assign(work=zone_of(work))[["uid", "work"]], on="uid"
)
pairs = pairs[pairs["home"] != pairs["work"]]
print(len(pairs), "people with different home and work zones")
pairs.groupby("home").size().describe().round(0)

27420 people with different home and work zones


count      36.0
mean      762.0
std       434.0
min        94.0
25%       364.0
50%       710.0
75%      1167.0
max      1649.0
dtype: float64

The sample has 27,420 people, 762 per origin zone on average, from 94 to 1,649 depending on the zone. The sampling rate grows with the zone number, as designed.

## Weighting the sample

A person from a zone with many sampled people stands for fewer commuters than a person from a zone with few. The expansion weight of everyone living in zone *h* is `residents(h) / sampled(h)`: the number of resident workers in the reference table over the number of sampled people. Summing the weights of the people of each pair of zones gives the estimated flow. As a baseline we use the plain counts, multiplied by one constant so that the total matches (otherwise the flows are on the scale of the sample and cannot be compared in absolute terms).

In [5]:
residents = zones.set_index("tile_id")["tot_outflow"]
sampled = pairs.groupby("home").size()
pairs = pairs.assign(weight=pairs["home"].map(residents / sampled))
constant = reference["flow"].sum() / len(pairs)


def flow_table(weights):
    summed = (
        pairs.assign(flow=weights)
        .groupby(["home", "work"], as_index=False)["flow"]
        .sum()
    )
    return summed.rename(columns={"home": "origin", "work": "destination"})


def as_flows(table):
    return FlowDataFrame(table, locations=locations, tile_id="tile_id")


estimates = {
    "unweighted": as_flows(flow_table(constant)),
    "weighted": as_flows(flow_table(pairs["weight"])),
}
pairs["weight"].describe().round(0)

count    27420.0
mean        67.0
std         31.0
min         43.0
25%         46.0
50%         58.0
75%         70.0
max        264.0
Name: weight, dtype: float64

The weights run from 43 to 264 commuters per person, with a median of 58: a person in a lightly sampled zone stands for many more commuters than one in a heavily sampled zone. The constant used for the unweighted baseline is 67 commuters per person (the average weight).

## Agreement with the reference

`compare_flow_models` scores each estimate against the reference over the union of the origin-destination pairs. The common part of commuters (CPC) is the share of commuters that both tables send along the same pair of zones (1 is identical); R-squared and the rank correlation compare the pair flows; the Poisson deviance and the pseudo R-squared treat the flows as counts.

In [6]:
reference_flows = as_flows(reference[["origin", "destination", "flow"]])
scores = compare_flow_models(reference_flows, estimates).to_pandas().set_index("model")
scores[["n_pairs", "cpc", "cpl", "r_squared", "spearman", "pseudo_r_squared"]].round(3)

,n_pairs,cpc,cpl,r_squared,spearman,pseudo_r_squared
model,,,,,,
unweighted,1260,0.831,0.966,0.857,0.901,0.605
weighted,1260,0.924,0.966,0.977,0.949,0.721


Weighting raises every score. The CPC goes from 0.83 to 0.92 (92% of the commuters are placed on the same pair of zones as in the reference), the R-squared of the pair flows from 0.86 to 0.98 and the rank correlation from 0.90 to 0.95. The CPL is 0.97 for both, because the sets of pairs with a flow are the same: weighting changes the sizes of the flows, not which pairs have flow. The pseudo R-squared is much lower than the R-squared because it measures deviations against Poisson noise, which is small for flows of this size.

## The distance decay, with a confidence interval

`Gravity.fit` fits the model by Poisson regression and `fit_summary` returns each parameter with a robust standard error, a 95% confidence interval and a p-value. We fit the reference table itself (to check the method), the unweighted estimate and the weighted one. The singly constrained model has one free level per origin, so a bias that depends only on the origin zone, like ours, cannot change the distance exponent.

In [7]:
rows = []
for name, flows in {"reference": reference_flows, **estimates}.items():
    model = Gravity(gravity_type="singly constrained")
    model.fit(flows)
    fit = model.fit_summary().to_pandas().set_index("parameter")
    stats = model.fit_statistics()
    rows.append(
        {
            "table": name,
            **fit.loc["distance_coefficient", ["estimate", "ci_low", "ci_high"]],
            "dispersion": stats["dispersion"],
            "pseudo_r_squared": stats["pseudo_r_squared"],
        }
    )
pd.DataFrame(rows).set_index("table").round(3)

,estimate,ci_low,ci_high,dispersion,pseudo_r_squared
table,,,,,
reference,-1.800,-1.803,-1.798,0.964,1.000
unweighted,-1.787,-1.811,-1.764,67.998,0.973
weighted,-1.773,-1.799,-1.747,85.609,0.961


The reference table gives -1.800 with an interval of -1.803 to -1.798, so the method recovers the exponent that generated it (the dispersion of 0.96 is the Poisson value). The two sampled tables give -1.787 (interval -1.811 to -1.764) and -1.773 (-1.799 to -1.747): within 0.03 of the truth, with the first interval containing -1.8 and the second just missing it. The weighted fit is not closer than the unweighted one here. The distance exponent does not need the weights, because the sampling bias depends on the origin and the model has one free level per origin; the weights matter for the levels, as the scores above show. The sampled tables are far from Poisson (dispersions of 68 and 86), which is why the robust intervals are wider than the model-based ones would be.

## The same on real trajectories

For real data there is no census table, but the two questions are the same: which distance decay do the flows show, and which model reproduces them better? We take the Brightkite check-ins, group them into H3 cells of about 22 km (resolution 4), keep the 60 busiest cells and the check-in-to-check-in trips between them, fit a singly constrained gravity model (relevance: check-ins in the cell) and generate the radiation model with the observed outflows of each cell.

In [8]:
from fastmob.data import load_brightkite

raw = load_brightkite(max_rows=300_000).to_pandas()
raw = raw[~(((raw["lat"] == 0) & (raw["lng"] == 0)) | (raw["lat"].abs() >= 90))]
counts = raw.groupby("uid").size()
checkins = (
    raw[raw["uid"].isin(counts[counts >= 100].index)]
    .sort_values(["uid", "datetime"])
    .reset_index(drop=True)
)

cells = fastmob.latlng_to_h3(checkins, resolution=4).rename(
    columns={"h3_cell": "tile_id"}
)
busiest = cells.groupby("tile_id").size().rename("relevance").nlargest(60)
trips = fastmob.trajectory_to_od(checkins, resolution=4).rename(
    columns={"count": "flow"}
)
trips = trips[
    trips["origin"].isin(busiest.index)
    & trips["destination"].isin(busiest.index)
    & (trips["origin"] != trips["destination"])
]
centres = fastmob.h3_to_latlng(pd.DataFrame({"h3_cell": busiest.index})).rename(
    columns={"h3_cell": "tile_id", "center_lat": "lat", "center_lng": "lng"}
)
cells_table = (
    centres.join(busiest, on="tile_id")
    .join(trips.groupby("origin")["flow"].sum().rename("tot_outflow"), on="tile_id")
    .fillna({"tot_outflow": 0})
)
real_locations = Locations.from_tessellation(
    cells_table, tile_id_col="tile_id", lat_col="lat", lng_col="lng"
)
observed = FlowDataFrame(
    trips.reset_index(drop=True), locations=real_locations, tile_id="tile_id"
)
print(
    len(cells_table),
    "cells,",
    len(trips),
    "pairs with flow,",
    int(trips["flow"].sum()),
    "trips",
)

60 cells, 646 pairs with flow, 18751 trips


In [9]:
from fastmob.models import Radiation

gravity = Gravity(gravity_type="singly constrained")
gravity.fit(observed)
print(gravity.fit_summary().to_pandas().round(3).to_string(index=False))
print(
    {
        k: round(v, 3) if isinstance(v, float) else v
        for k, v in gravity.fit_statistics().items()
    }
)

models = {
    "gravity": gravity.generate(real_locations, out_format="flows"),
    "radiation": Radiation().generate(real_locations, out_format="flows"),
}
compare_flow_models(observed, models).to_pandas().set_index("model").round(3)

           parameter  estimate  std_error  ci_low  ci_high  p_value
     destination_exp     0.642      0.044   0.555    0.728      0.0
distance_coefficient    -1.208      0.032  -1.271   -1.146      0.0


{'n_observations': 646, 'log_likelihood': -4027.998, 'deviance': 6061.76, 'pseudo_r_squared': 0.933, 'dispersion': 15.835, 'robust_errors': True}


,n_pairs,observed_flow,model_flow,cpc,cpl,r_squared,spearman,poisson_deviance,pseudo_r_squared
model,,,,,,,,,
gravity,3304,18751.0,18751.0,0.779,0.327,0.962,0.498,10252.294,0.933
radiation,1018,18751.0,18629.0,0.587,0.534,0.763,0.060,58538.019,0.457


The distance exponent of the check-in flows is -1.21 (95% interval -1.27 to -1.15) and the destination exponent is 0.64 (0.56 to 0.73), from 646 pairs of cells. The dispersion is 15.8, far above 1, which confirms that the robust intervals are needed. The gravity model reproduces the flows better than the radiation model on every score except the CPL: CPC 0.78 against 0.59, R-squared 0.96 against 0.76, rank correlation 0.50 against 0.06 and pseudo R-squared 0.93 against 0.46. Radiation has a higher CPL (0.53 against 0.33) because it predicts fewer pairs (1,018 against 3,304 for the gravity model, which gives every pair some flow). Both rank correlations are low because most of the 3,304 pairs have no observed flow. These are 60 cells of 22 km in a sample of check-ins, so the numbers describe this data set, not human mobility in general.

## Interpretation and limits

- The reference table here is synthetic, which is what lets us check that the method recovers the truth. With an official table the check is only as good as that table: commuting tables count where people work or live by registration, and sampled trajectories can include second homes, remote work or night-shift work.
- The expansion weight assumes that, within a home zone, the people sampled are like the other residents. A bias that depends on who works where (not only on where people live) is not corrected, and shows as a lower CPC.
- Home and work are inferred. Change `start_night`, `end_night`, `start_work` and `end_work`, or the number of nights required, and see whether the flows change.
- CPC and the other scores depend on the size of the zones: finer zones give lower values. Report the zone size with the score.
- The Poisson regression assumes counts. Real flows are overdispersed (the dispersion above is far above 1), so use the robust intervals that `fit_summary` reports, and treat p-values as approximate.
- A gravity fit on a few dozen cells describes those cells. The distance exponent depends on the range of distances and on the zone size.

## Next steps

- Fit and compare the distributions of jump lengths with intervals and alternatives: [Compare to mobility laws](compare-to-mobility-laws.md).
- How the fits are estimated and what to report: [How the fits work](fitting-methods.md).
- Score a flow model with more metrics: [Fit and compare distributions](fit-and-evaluate.md).

## Related API

[`home_location`](../reference/measures/individual.md#fastmob.measures.individual.home_location.home_location) ·
[`work_location`](../reference/measures/individual.md#fastmob.measures.individual.work_location.work_location) ·
[`Gravity.fit`](../reference/models.md#fastmob.models.gravity.Gravity.fit) ·
[`Gravity.fit_summary`](../reference/models.md#fastmob.models.gravity.Gravity.fit_summary) ·
[`Gravity.fit_statistics`](../reference/models.md#fastmob.models.gravity.Gravity.fit_statistics) ·
[`Radiation.generate`](../reference/models.md#fastmob.models.radiation.Radiation.generate) ·
[`compare_flow_models`](../reference/measures/evaluation.md#fastmob.measures.evaluation.compare_flow_models)